# NB13b — Nivel 2: Prior de Cohorte RUNA v2 (eligibility relajada)

**Pregunta de investigación (Q2 — parte 1):**
> ¿Puede un modelo entrenado con la cohorte RUNA propia (atletas reales con consentimiento) mejorar el prior poblacional (N1) en la estimación del ritmo FC↔pace?

## Contexto

El **Nivel 2** es el núcleo generalizable de la arquitectura jerárquica RUNA. Aprende de las sesiones reales de la cohorte propia, usando el prior poblacional N1 como *stacking feature* (Wolpert 1992).  
La unidad de análisis es **cada sesión individual** — no requiere historial acumulado del atleta (eso es N3).

**Diferencia clave vs N1:**
- N1 aprendió de Endomondo/FitRec (internet, 20,710 sesiones, 356 usuarios)
- N2 aprende de la cohorte RUNA (atletas reales reclutados activamente con consentimiento académico)

## Features N2 (16)
| Feature | Fuente | Tipo |
|---------|--------|------|
| `pred_nivel1` | N1 Ridge inference | Stacking (Wolpert) |
| `age`, `sex_bin` | Formulario onboarding | Demografía |
| `vdot` | Mejor PR declarado (Daniels 1998) | Fitness proxy |
| `avg_hr`, `pct_hrmax`, `zona_hr` | Supabase activities.raw | Intensidad sesión |
| `log_distance_km` | Supabase activities | Volumen |
| `elevation_m`, `has_elevation` | Supabase activities | Terreno |
| `cadence_filled` | Supabase activities.raw | Economía |
| `dow_sin/cos`, `month_sin/cos` | Fecha actividad | Estacionalidad |

**Target:** `pace_sec_per_km` = duration_sec / distance_km

**Excluidos de N2 (→ N3):** CTL, ATL, TSB, ACWR (requieren historial acumulado)  
**Excluidos por sesgo:** PRs como features directas (sesgo de recuerdo; solo se usan para VDOT)

## Protocolo
- **Validación:** LOAO-CV (Leave-One-Athlete-Out) — 1 fold por atleta elegible
- **Sample weighting:** w = 1/n_sesiones_atleta (evita dominancia de atletas prolíficos)
- **Comparación:** 7 familias sklearn + prueba Friedman-Nemenyi
- **Elegibilidad N2:** ≥3 runs con HR + ≥2 semanas de historial

## Índice
1. Setup & carga N1
2. Carga de datos desde Supabase
3. Feature engineering
4. EDA (distribuciones, correlaciones, demografía)
5. Ridge LOAO-CV — baseline N2
6. AutoML manual — 7 familias LOAO-CV
7. Friedman-Nemenyi
8. Modelo ganador — análisis y serialización

---
## 0. Setup & Imports

In [ ]:
import json
import os
import sys
import warnings
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy.stats import friedmanchisquare, spearmanr

from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import LinearSVR
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error

try:
    import scikit_posthocs as sp
    HAS_POSTHOCS = True
except ImportError:
    HAS_POSTHOCS = False
    print('scikit-posthocs no disponible — Nemenyi se implementara manualmente')

try:
    from dotenv import load_dotenv
    load_dotenv(Path('../../.env'))
    print('dotenv cargado desde ../../.env')
except ImportError:
    print('python-dotenv no instalado — asegura que SUPABASE_URL y SUPABASE_SERVICE_KEY esten en el entorno')

from supabase import create_client

SUPABASE_URL = os.environ.get('SUPABASE_URL', '')
SUPABASE_KEY = os.environ.get('SUPABASE_SERVICE_KEY', '')

if not SUPABASE_URL or not SUPABASE_KEY:
    raise EnvironmentError('SUPABASE_URL o SUPABASE_SERVICE_KEY no configurados. Verifica .env')

sb = create_client(SUPABASE_URL, SUPABASE_KEY)

warnings.filterwarnings('ignore')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 11
sns.set_theme(style='whitegrid')

FIGURES_DIR = Path('../figures')
FIGURES_DIR.mkdir(exist_ok=True)

RANDOM_STATE = 42
PAGE = 1000

print('Setup OK ✓')
print(f'Supabase URL: {SUPABASE_URL[:40]}...')

In [ ]:
# ── Cargar modelo N1 (Ridge) para stacking ────────────────────────────────────
N1_PATH = Path('../../api/models/nivel1_ridge_v4.json')

with open(N1_PATH) as f:
    n1 = json.load(f)

N1_FEATURES  = n1['features']
N1_COEFS     = np.array(n1['coefs'])
N1_INTERCEPT = n1['intercept']
N1_MEAN      = np.array(n1['scaler_mean'])
N1_SCALE     = np.array(n1['scaler_scale'])
N1_MAE       = n1['mae_sec_km']

def predict_n1_sec_km(row: dict) -> float:
    '''Predice pace_sec_per_km usando el modelo N1 Ridge (nivel1_ridge_v4).
    Input: dict con keys = N1_FEATURES
    Output: pace en sec/km (float)
    '''
    feats = np.array([row[k] for k in N1_FEATURES], dtype=float)
    feats_scaled = (feats - N1_MEAN) / N1_SCALE
    pace_min_km = float(np.dot(N1_COEFS, feats_scaled) + N1_INTERCEPT)
    return pace_min_km * 60.0  # min/km → sec/km

print(f'N1 model: {N1_PATH.name}')
print(f'  Entrenado en: {n1["trained_on"]}')
print(f'  MAE CV: {N1_MAE:.1f} sec/km ({N1_MAE/60:.2f} min/km)')
print(f'  Features: {N1_FEATURES}')

---
## 1. Carga de datos desde Supabase

In [ ]:
# ── 1.1 Cargar actividades Run/TrailRun CON HR (paginado) ─────────────────────
# Supabase max_rows = 100,000 → paginamos por si acaso
print('Cargando actividades Run/TrailRun con HR desde Supabase...')

hr_rows = []
offset  = 0

while True:
    batch = (
        sb.table('activities')
        .select('strava_id,cedula,activity_date,distance_m,duration_sec,elevation_m,raw')
        .in_('sport_type', ['Run', 'TrailRun'])
        .not_.is_('raw->>average_heartrate', 'null')
        .order('strava_id')
        .range(offset, offset + PAGE - 1)
        .execute()
        .data or []
    )
    hr_rows.extend(batch)
    if len(batch) < PAGE:
        break
    offset += PAGE
    if offset % 5000 == 0:
        print(f'  ...{offset:,} filas cargadas')

print(f'Total actividades HR cargadas: {len(hr_rows):,}')
print(f'Atletas unicos en dataset raw: {len(set(r["cedula"] for r in hr_rows)):,}')

In [ ]:
# ── 1.2 Extraer campos del JSONB raw y calcular target ────────────────────────
df_raw = pd.DataFrame(hr_rows)

def safe_raw(raw_val, key, default=np.nan):
    '''Extrae campo de raw JSONB (puede llegar como dict o string).'''
    if isinstance(raw_val, str):
        try:
            raw_val = json.loads(raw_val)
        except Exception:
            return default
    if isinstance(raw_val, dict):
        return raw_val.get(key, default)
    return default

df_raw['avg_hr']     = df_raw['raw'].apply(lambda r: safe_raw(r, 'average_heartrate'))
df_raw['max_hr_obs'] = df_raw['raw'].apply(lambda r: safe_raw(r, 'max_heartrate'))
df_raw['cadence']    = df_raw['raw'].apply(lambda r: safe_raw(r, 'average_cadence'))
df_raw = df_raw.drop(columns=['raw'])

# Coerciones numéricas
for col in ['avg_hr', 'max_hr_obs', 'cadence', 'distance_m', 'duration_sec', 'elevation_m']:
    df_raw[col] = pd.to_numeric(df_raw[col], errors='coerce')

df_raw['distance_km']       = df_raw['distance_m'] / 1000.0
df_raw['pace_sec_per_km']   = df_raw['duration_sec'] / df_raw['distance_km']
df_raw['elevation_m']       = df_raw['elevation_m'].fillna(0.0)
df_raw['activity_date']     = pd.to_datetime(df_raw['activity_date'], utc=True, errors='coerce')

# ── Filtros de calidad ────────────────────────────────────────────────────────
n_before = len(df_raw)
df_act = df_raw[
    df_raw['avg_hr'].notna() &
    (df_raw['avg_hr'].between(50, 220)) &
    (df_raw['distance_km'] > 1.0) &
    (df_raw['pace_sec_per_km'].between(200, 1000)) &  # 3:20–16:40 min/km
    df_raw['duration_sec'].notna() &
    df_raw['activity_date'].notna()
].copy()

print(f'Filas antes del filtro: {n_before:,}')
print(f'Filas después del filtro de calidad: {len(df_act):,}  (-{n_before-len(df_act):,})')
print(f'Cadencia disponible: {df_act["cadence"].notna().sum():,} ({df_act["cadence"].notna().mean()*100:.0f}%)')

In [ ]:
# ── 1.3 Determinar atletas N2 elegibles ──────────────────────────────────────
stats = (
    df_act.groupby('cedula')
    .agg(
        n_hr_runs   = ('strava_id', 'count'),
        first_date  = ('activity_date', 'min'),
        last_date   = ('activity_date', 'max'),
    )
    .reset_index()
)
stats['weeks_span'] = ((stats['last_date'] - stats['first_date']).dt.days / 7.0).round(1)

# Criterios N2: >= 3 HR runs Y >= 2 semanas (all-time, sin filtro de PROJECT_START)
N2_MIN_RUNS  = 5
N2_MIN_WEEKS = 4

stats['n2_eligible'] = (stats['n_hr_runs'] >= N2_MIN_RUNS) & (stats['weeks_span'] >= N2_MIN_WEEKS)
eligible_cedulas = stats.loc[stats['n2_eligible'], 'cedula'].tolist()

print(f'Atletas totales con HR runs: {len(stats)}')
print(f'N2 elegibles (>={N2_MIN_RUNS} HR runs + >={N2_MIN_WEEKS} semanas): {len(eligible_cedulas)}')
print()
print('Distribución de sesiones en elegibles:')
elig_stats = stats[stats['n2_eligible']]
print(elig_stats['n_hr_runs'].describe().round(1))
print()
print('Distribución weeks_span en elegibles:')
print(elig_stats['weeks_span'].describe().round(1))

In [ ]:
# ── 1.4 Cargar perfiles de atletas elegibles desde Supabase ──────────────────
print(f'Cargando perfiles de {len(eligible_cedulas)} atletas elegibles...')

prof_rows = (
    sb.table('athlete_profiles')
    .select('cedula,raw')
    .in_('cedula', eligible_cedulas)
    .execute()
    .data or []
)

# Parsear raw JSONB
profiles = {}
for row in prof_rows:
    ced = row['cedula']
    raw = row.get('raw') or {}
    if isinstance(raw, str):
        try:
            raw = json.loads(raw)
        except Exception:
            raw = {}
    profiles[ced] = raw

print(f'Perfiles encontrados: {len(profiles)}/{len(eligible_cedulas)}')
sin_perfil = [c for c in eligible_cedulas if c not in profiles]
if sin_perfil:
    print(f'Sin perfil en Supabase (solo Strava, sin onboarding): {sin_perfil}')

# Extraer campos relevantes
demo_list = []
for ced in eligible_cedulas:
    p = profiles.get(ced, {})
    age  = p.get('age')
    sex  = p.get('sex') or p.get('gender') or p.get('sexo') or p.get('genero')
    sex_bin = 1 if sex and str(sex).strip().upper() in ('M', 'MALE', 'MASCULINO', 'HOMBRE', 'H') else 0
    # PRs — varios nombres posibles según versión del formulario
    pr_5k  = p.get('pr_5k_sec')  or p.get('pr_5k')
    pr_10k = p.get('pr_10k_sec') or p.get('pr_10k')
    pr_21k = p.get('pr_21k_sec') or p.get('pr_21k') or p.get('pr_half_sec') or p.get('pr_media_maraton_sec')
    pr_42k = p.get('pr_42k_sec') or p.get('pr_42k') or p.get('pr_marathon_sec') or p.get('pr_maraton_sec')
    demo_list.append({'cedula': ced, 'age': age, 'sex_bin': sex_bin,
                      'pr_5k_sec': pr_5k, 'pr_10k_sec': pr_10k,
                      'pr_21k_sec': pr_21k, 'pr_42k_sec': pr_42k})

df_demo = pd.DataFrame(demo_list)
for col in ['age', 'pr_5k_sec', 'pr_10k_sec', 'pr_21k_sec', 'pr_42k_sec']:
    df_demo[col] = pd.to_numeric(df_demo[col], errors='coerce')

print()
print('Demographics:')
print(f'  Con edad: {df_demo["age"].notna().sum()}/{len(df_demo)}')
print(f'  Masculino: {df_demo["sex_bin"].sum()} | Femenino: {(df_demo["sex_bin"]==0).sum()}')
print(f'  Con PR 5K: {df_demo["pr_5k_sec"].notna().sum()}')
print(f'  Con PR 10K: {df_demo["pr_10k_sec"].notna().sum()}')
print(f'  Con PR 21K: {df_demo["pr_21k_sec"].notna().sum()}')
print(f'  Con PR 42K: {df_demo["pr_42k_sec"].notna().sum()}')

---
## 2. Feature Engineering

In [ ]:
# ── 2.1 FCmax por atleta ──────────────────────────────────────────────────────
# Empirico: percentil 95 de max_hr_obs (robusto a outliers de sensor)
# Fallback: Tanaka (2001) → FCmax = 208 - 0.7 × edad
fcmax_emp = (
    df_act[df_act['cedula'].isin(eligible_cedulas)]
    .groupby('cedula')['max_hr_obs']
    .quantile(0.95)
    .reset_index()
    .rename(columns={'max_hr_obs': 'fcmax_emp'})
)

df_demo = df_demo.merge(fcmax_emp, on='cedula', how='left')
df_demo['fcmax_tanaka'] = (208.0 - 0.7 * df_demo['age'].fillna(35.0)).round(1)
df_demo['fcmax']        = df_demo['fcmax_emp'].fillna(df_demo['fcmax_tanaka'])

print('FCmax por atleta:')
print(df_demo[['cedula','age','fcmax_emp','fcmax_tanaka','fcmax']].to_string(index=False))

In [ ]:
# ── 2.2 VDOT estimado (Jack Daniels 1998) ────────────────────────────────────
# Fórmula cerrada: VDOT = VO2 / %VO2max
# Referencia: Daniels J. (1998). Daniels Running Formula.

def vdot_from_pr(distance_km: float, time_sec: float) -> float:
    '''Estima VDOT a partir de un PR (distancia en km, tiempo en seg).
    Devuelve NaN si los inputs son inválidos o el VDOT está fuera de rango fisiológico.
    '''
    if pd.isna(distance_km) or pd.isna(time_sec) or time_sec <= 0:
        return np.nan
    t_min = time_sec / 60.0
    v     = distance_km * 1000.0 / t_min         # velocidad en m/min
    pct   = (0.8 + 0.1894393  * np.exp(-0.012778  * t_min)
                 + 0.2989558  * np.exp(-0.1932605 * t_min))
    vo2   = -4.60 + 0.182258 * v + 0.000104 * (v ** 2)
    vdot  = vo2 / pct
    return float(vdot) if 15 < vdot < 95 else np.nan

def best_vdot(row) -> float:
    '''Usa el PR más largo disponible (más estable para VDOT).'''
    for dist, col in [(42.195, 'pr_42k_sec'), (21.0975, 'pr_21k_sec'),
                      (10.0,   'pr_10k_sec'), (5.0,    'pr_5k_sec')]:
        v = vdot_from_pr(dist, row.get(col))
        if not np.isnan(v):
            return v
    return np.nan

df_demo['vdot'] = df_demo.apply(best_vdot, axis=1)

print('VDOT estimado por atleta:')
print(df_demo[['cedula','age','sex_bin','fcmax','vdot']].to_string(index=False))
print()
print(f'Con VDOT estimado: {df_demo["vdot"].notna().sum()}/{len(df_demo)}')
print(f'VDOT — media: {df_demo["vdot"].mean():.1f}  std: {df_demo["vdot"].std():.1f}')
print(f'  (Referencia: corredor promedio ~45, sub-3h ~58+, elite >70)')

In [ ]:
# ── 2.3 Feature engineering de sesiones ──────────────────────────────────────
df_n2 = df_act[df_act['cedula'].isin(eligible_cedulas)].copy()
df_n2 = df_n2.merge(df_demo[['cedula','age','sex_bin','fcmax','vdot']], on='cedula', how='left')

# --- Intensidad cardíaca ---
df_n2['pct_hrmax'] = (df_n2['avg_hr'] / df_n2['fcmax'] * 100.0).round(2)

def assign_zone(pct: float) -> int:
    if pd.isna(pct):  return 0
    if pct < 60:      return 1
    if pct < 70:      return 2
    if pct < 80:      return 3
    if pct < 90:      return 4
    return 5

df_n2['zona_hr'] = df_n2['pct_hrmax'].apply(assign_zone)

# --- Volumen y terreno ---
df_n2['log_distance_km'] = np.log1p(df_n2['distance_km'])
df_n2['has_elevation']   = (df_n2['elevation_m'] > 50).astype(int)

# --- Estacionalidad (encoding cíclico) ---
df_n2['dow']       = df_n2['activity_date'].dt.dayofweek    # 0=lunes
df_n2['month']     = df_n2['activity_date'].dt.month
df_n2['dow_sin']   = np.sin(2 * np.pi * df_n2['dow']         / 7.0)
df_n2['dow_cos']   = np.cos(2 * np.pi * df_n2['dow']         / 7.0)
df_n2['month_sin'] = np.sin(2 * np.pi * (df_n2['month'] - 1) / 12.0)
df_n2['month_cos'] = np.cos(2 * np.pi * (df_n2['month'] - 1) / 12.0)

# --- Variables auxiliares para N1 stacking ---
df_n2['log_duration'] = np.log(df_n2['duration_sec'].clip(lower=1))
df_n2['hr_max_rel']   = df_n2['avg_hr'] / df_n2['fcmax']
df_n2['dens_hr']      = df_n2['avg_hr'] / df_n2['log_duration']

# --- Cadencia: imputar mediana por atleta, luego mediana global ---
cadence_by_ath = df_n2.groupby('cedula')['cadence'].transform('median')
cadence_global = df_n2['cadence'].median()
df_n2['cadence_filled'] = df_n2['cadence'].fillna(cadence_by_ath).fillna(cadence_global)
df_n2['has_cadence']    = df_n2['cadence'].notna().astype(int)

print(f'Sesiones tras feature engineering: {len(df_n2):,}')
print(f'Distribución zona HR:')
print(df_n2['zona_hr'].value_counts().sort_index().rename({1:'Z1',2:'Z2',3:'Z3',4:'Z4',5:'Z5'}))

In [ ]:
# ── 2.4 N1 stacking: predicción del prior poblacional por sesión ──────────────
print('Calculando predicciones N1 (stacking)...')

def compute_n1_pred(row) -> float:
    try:
        feat_dict = {
            'gender_bin':  row['sex_bin'],
            'fcmax_obs':   row['fcmax'],
            'hr_mean':     row['avg_hr'],
            'pct_fcmax':   row['pct_hrmax'],
            'zona_num':    row['zona_hr'],
            'hr_max_rel':  row['hr_max_rel'],
            'log_duration': row['log_duration'],
            'dens_hr':     row['dens_hr'],
        }
        return predict_n1_sec_km(feat_dict)
    except Exception:
        return np.nan

df_n2['pred_nivel1'] = df_n2.apply(compute_n1_pred, axis=1)

n_ok = df_n2['pred_nivel1'].notna().sum()
print(f'N1 preds OK: {n_ok:,}/{len(df_n2):,}')

# Diagnóstico: error N1 sobre este conjunto (upper bound que N2 debe superar)
valid_mask = df_n2['pred_nivel1'].notna() & df_n2['pace_sec_per_km'].notna()
mae_n1_on_n2_data = mean_absolute_error(
    df_n2.loc[valid_mask, 'pace_sec_per_km'],
    df_n2.loc[valid_mask, 'pred_nivel1']
)
print(f'N1 MAE en datos RUNA (referencia a superar): {mae_n1_on_n2_data:.1f} sec/km ({mae_n1_on_n2_data/60:.2f} min/km)')
print(f'  (N1 MAE en entrenamiento Endomondo: {N1_MAE:.1f} sec/km)')
print(f'  → Si MAE N2 > MAE N1: la cohorte RUNA tiene características distintas a Endomondo')

In [ ]:
# ── 2.5 Dataset final ML + sample weights ────────────────────────────────────
FEATURES = [
    'pred_nivel1',       # stacking N1
    'age',               # demografía
    'sex_bin',
    'vdot',              # fitness proxy (Daniels)
    'avg_hr',            # intensidad
    'pct_hrmax',
    'zona_hr',
    'log_distance_km',   # volumen
    'elevation_m',       # terreno
    'has_elevation',
    'cadence_filled',    # economía
    'dow_sin', 'dow_cos',        # estacionalidad
    'month_sin', 'month_cos',
]

TARGET = 'pace_sec_per_km'

# Drop filas con NaN en features críticas
# Nota: 'fcmax' no está en FEATURES (es intermedio), usamos 'pct_hrmax'
# que es NaN exactamente cuando fcmax es NaN (pct_hrmax = avg_hr / fcmax * 100)
critical = ['age', 'pred_nivel1', TARGET, 'pct_hrmax']
df_ml = df_n2[FEATURES + [TARGET, 'cedula']].dropna(subset=critical).copy()

# VDOT faltante: imputar con mediana de la cohorte
vdot_median = df_ml['vdot'].median()
df_ml['vdot'] = df_ml['vdot'].fillna(vdot_median)
print(f'VDOT imputado con mediana {vdot_median:.1f} para {df_ml["vdot"].isna().sum()} sesiones')

# Sample weights: w = 1 / n_sesiones_atleta (evita dominancia de atletas con muchas sesiones)
n_per_ath = df_ml.groupby('cedula').size().rename('n_sessions')
df_ml = df_ml.join(n_per_ath, on='cedula')
df_ml['sample_weight'] = 1.0 / df_ml['n_sessions']

athletes_ml = df_ml['cedula'].unique()
n_athletes  = len(athletes_ml)
n_sessions  = len(df_ml)

print(f'\nDataset ML final:')
print(f'  Atletas: {n_athletes} (folds LOAO-CV)')
print(f'  Sesiones totales: {n_sessions:,}')
print(f'  Features: {len(FEATURES)}')
print(f'  Target: {TARGET}')
print(f'\nSesiones por atleta:')
print(df_ml.groupby('cedula').size().describe().round(1))
print(f'\nNaN residuales en features:')
nans = df_ml[FEATURES].isnull().sum()
print(nans[nans > 0] if nans.any() else '  Ninguno ✓')

---
## 3. EDA — Análisis Exploratorio

In [ ]:
# ── 3.1 Distribución de sesiones por atleta (imbalance) ──────────────────────
sess_counts = df_ml.groupby('cedula').size().sort_values(ascending=False).reset_index()
sess_counts.columns = ['cedula', 'n_sessions']

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Histograma de sesiones
ax = axes[0]
ax.bar(range(len(sess_counts)), sess_counts['n_sessions'], color='steelblue', edgecolor='white')
ax.axhline(sess_counts['n_sessions'].mean(), color='red', linestyle='--', lw=1.5,
           label=f'Media = {sess_counts["n_sessions"].mean():.0f}')
ax.set_xlabel('Atleta (ordenado por # sesiones)')
ax.set_ylabel('# Sesiones con HR')
ax.set_title(f'Distribución de sesiones por atleta\n({n_athletes} atletas elegibles N2)')
ax.legend()

# Curva de Lorenz (imbalance)
ax = axes[1]
sorted_w = np.sort(1.0 / sess_counts['n_sessions'].values)[::-1]
sorted_w_norm = sorted_w / sorted_w.sum()
cumsum = np.cumsum(sorted_w_norm)
ax.plot(np.linspace(0, 1, len(cumsum)), cumsum, color='steelblue', lw=2, label='Con sample_weight')
sorted_n = np.sort(sess_counts['n_sessions'].values)[::-1]
sorted_n_norm = sorted_n / sorted_n.sum()
cumsum_raw = np.cumsum(sorted_n_norm)
ax.plot(np.linspace(0, 1, len(cumsum_raw)), cumsum_raw, color='tomato', lw=2, label='Sin weight (raw)')
ax.plot([0, 1], [0, 1], 'k--', lw=1, label='Igualdad perfecta')
ax.set_xlabel('Fracción de atletas')
ax.set_ylabel('Fracción de peso acumulado')
ax.set_title('Efecto del sample weighting\n(w = 1/n_sesiones_atleta)')
ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig(FIGURES_DIR / '13_eda_sesiones_imbalance.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Coef. variación (imbalance): {sess_counts["n_sessions"].std()/sess_counts["n_sessions"].mean():.2f}')
print(f'  (0 = balance perfecto, >1 = alto imbalance)')

In [ ]:
# ── 3.2 Correlación HR↔Pace (clave del modelo) ───────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Global: scatter pct_hrmax vs pace
ax = axes[0]
sample = df_ml.sample(min(5000, len(df_ml)), random_state=RANDOM_STATE)
sc = ax.scatter(sample['pct_hrmax'], sample['pace_sec_per_km'] / 60,
                alpha=0.25, s=8, c=sample['zona_hr'], cmap='RdYlGn_r')
plt.colorbar(sc, ax=ax, label='Zona HR')
r_global, _ = spearmanr(df_ml['pct_hrmax'].dropna(), df_ml['pace_sec_per_km'].dropna())
ax.set_xlabel('% FCmax')
ax.set_ylabel('Pace (min/km)')
ax.set_title(f'% FCmax vs Pace\n(ρ Spearman = {r_global:.3f}, N = {len(df_ml):,})')
ax.invert_yaxis()

# Por atleta: correlacion individual
ax = axes[1]
ath_corrs = []
for ced, grp in df_ml.groupby('cedula'):
    if len(grp) >= 3:
        r, _ = spearmanr(grp['pct_hrmax'], grp['pace_sec_per_km'])
        ath_corrs.append({'cedula': ced, 'rho': r, 'n': len(grp)})
ath_corrs_df = pd.DataFrame(ath_corrs).sort_values('rho')
colors = ['tomato' if r > 0 else 'steelblue' for r in ath_corrs_df['rho']]
ax.barh(range(len(ath_corrs_df)), ath_corrs_df['rho'], color=colors, edgecolor='white', height=0.8)
ax.axvline(0, color='black', lw=1)
ax.axvline(ath_corrs_df['rho'].median(), color='orange', lw=2, linestyle='--',
           label=f'Mediana = {ath_corrs_df["rho"].median():.3f}')
ax.set_xlabel('ρ Spearman (% FCmax, pace)')
ax.set_title(f'Correlación por atleta\n(positivo = a más FC, más lento = esperado)')
ax.legend(fontsize=9)

# N1 pred vs actual (calibración del prior)
ax = axes[2]
ax.scatter(df_ml['pred_nivel1'] / 60, df_ml['pace_sec_per_km'] / 60,
           alpha=0.15, s=6, color='steelblue')
lims = [df_ml['pace_sec_per_km'].min()/60 - 0.5, df_ml['pace_sec_per_km'].max()/60 + 0.5]
ax.plot(lims, lims, 'k--', lw=1.5, label='Predicción perfecta')
ax.set_xlabel('N1 predicho (min/km)')
ax.set_ylabel('Pace real (min/km)')
ax.set_title(f'Calibración N1 sobre cohorte RUNA\nMAE = {mae_n1_on_n2_data:.1f} sec/km')
ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig(FIGURES_DIR / '13_eda_correlaciones.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Correlación positiva HR↔pace en {(ath_corrs_df["rho"] > 0).sum()}/{len(ath_corrs_df)} atletas')
print(f'  (esperado: prácticamente todos — más FC = más lento = correlación positiva)')

In [ ]:
# ── 3.3 Demografía y distribución del target ─────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Distribución de pace (target)
ax = axes[0]
df_ml['pace_min_km'] = df_ml[TARGET] / 60.0
ax.hist(df_ml['pace_min_km'], bins=50, color='steelblue', edgecolor='white')
ax.axvline(df_ml['pace_min_km'].mean(), color='red', linestyle='--',
           label=f'Media = {df_ml["pace_min_km"].mean():.2f} min/km')
ax.set_xlabel('Pace (min/km)')
ax.set_ylabel('Frecuencia')
ax.set_title('Distribución del target\n(pace por sesión)')
ax.legend(fontsize=9)

# Distribución de edad
ax = axes[1]
age_dist = df_demo[df_demo['cedula'].isin(athletes_ml)]['age'].dropna()
ax.hist(age_dist, bins=15, color='salmon', edgecolor='white')
ax.set_xlabel('Edad')
ax.set_ylabel('# Atletas')
ax.set_title(f'Distribución de edad\nCohorte N2 ({len(athletes_ml)} atletas)')
ax.axvline(age_dist.mean(), color='red', linestyle='--',
           label=f'Media = {age_dist.mean():.0f} años')
ax.legend(fontsize=9)

# Pace por zona HR (box plot)
ax = axes[2]
zone_data = [df_ml[df_ml['zona_hr'] == z]['pace_min_km'].values for z in [1, 2, 3, 4, 5]]
zone_labels = [f'Z{z}\n(n={len(d):,})' for z, d in enumerate(zone_data, 1)]
bp = ax.boxplot(zone_data, labels=zone_labels, patch_artist=True, showfliers=False)
colors_zones = ['#4575b4', '#91bfdb', '#fee090', '#fc8d59', '#d73027']
for patch, color in zip(bp['boxes'], colors_zones):
    patch.set_facecolor(color)
    patch.set_alpha(0.8)
ax.set_ylabel('Pace (min/km)')
ax.set_title('Pace por zona HR\n(señal de la variable clave)')
ax.invert_yaxis()

plt.tight_layout()
plt.savefig(FIGURES_DIR / '13_eda_demografias.png', dpi=150, bbox_inches='tight')
plt.show()

print('Estadísticos del target (pace_sec_per_km):')
print(df_ml[TARGET].describe().round(1))
print(f'\nTarget en min/km:')
print((df_ml[TARGET]/60).describe().round(2))

In [ ]:
# ── 3.4 Heatmap de correlaciones entre features ───────────────────────────────
fig, ax = plt.subplots(figsize=(12, 10))

corr_feats = FEATURES + [TARGET]
corr_matrix = df_ml[corr_feats].corr(method='spearman')

mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)
sns.heatmap(
    corr_matrix,
    mask=mask,
    annot=True, fmt='.2f', annot_kws={'size': 7},
    cmap='RdBu_r', center=0, vmin=-1, vmax=1,
    square=True, ax=ax,
    linewidths=0.5,
)
ax.set_title('Correlaciones Spearman — Features N2 + Target', pad=12)
plt.tight_layout()
plt.savefig(FIGURES_DIR / '13_eda_heatmap_corr.png', dpi=150, bbox_inches='tight')
plt.show()

print('Top correlaciones con el target (|ρ| > 0.3):')
target_corrs = corr_matrix[TARGET].drop(TARGET).abs().sort_values(ascending=False)
print(target_corrs[target_corrs > 0.3].round(3))

---
## 4. LOAO-CV — Ridge baseline N2

**Leave-One-Athlete-Out Cross-Validation (LOAO-CV):**  
- Para cada atleta k (k = 1..N): entrenar con todos los demás atletas, predecir sesiones del atleta k
- El fold de test siempre tiene sesiones de UN SOLO ATLETA que el modelo NUNCA vio
- Simula el escenario real: atleta nuevo llega sin historial propio
- Sample weights aplicados solo en TRAINING (no en evaluación)

Reportamos: **MAE por fold** (vector de N valores) → insumo para Friedman-Nemenyi.

In [ ]:
# ── 4.1 Función LOAO-CV reutilizable ─────────────────────────────────────────
def run_loao_cv(model, df: pd.DataFrame, features: list, target: str,
                use_weights: bool = True, verbose: bool = True) -> dict:
    '''LOAO-CV: devuelve MAE por fold (por atleta) y MAE global.
    Retorna: dict con keys [mae_per_fold, mae_global, n_folds, n_sessions]
    '''
    athletes = df['cedula'].unique()
    fold_maes = []
    fold_info = []

    for k, ath_test in enumerate(athletes):
        mask_train = df['cedula'] != ath_test
        mask_test  = df['cedula'] == ath_test

        X_train = df.loc[mask_train, features].values
        y_train = df.loc[mask_train, target].values
        X_test  = df.loc[mask_test,  features].values
        y_test  = df.loc[mask_test,  target].values

        if use_weights:
            w_train = df.loc[mask_train, 'sample_weight'].values
            model.fit(X_train, y_train, **{'model__sample_weight': w_train}
                      if hasattr(model, 'named_steps') else {})
        else:
            model.fit(X_train, y_train)

        y_pred = model.predict(X_test)
        mae_fold = mean_absolute_error(y_test, y_pred)
        fold_maes.append(mae_fold)
        fold_info.append({'cedula': ath_test, 'mae': mae_fold, 'n_test': mask_test.sum()})

        if verbose and (k + 1) % 10 == 0:
            print(f'  Fold {k+1}/{len(athletes)} done — running MAE: {np.mean(fold_maes):.1f} sec/km')

    mae_arr    = np.array(fold_maes)
    mae_global = mae_arr.mean()

    return {
        'mae_per_fold': mae_arr,
        'mae_global':   mae_global,
        'mae_std':      mae_arr.std(),
        'n_folds':      len(athletes),
        'n_sessions':   len(df),
        'fold_df':      pd.DataFrame(fold_info),
    }

print('Función LOAO-CV definida ✓')
print(f'Folds totales: {n_athletes} (uno por atleta elegible N2)')

In [ ]:
# ── 4.2 Ridge LOAO-CV — baseline N2 ─────────────────────────────────────────
# Ridge con StandardScaler — pipeline para evitar data leakage en el scaler
from sklearn.pipeline import Pipeline

ridge_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('model',  Ridge(alpha=1.0)),
])

print('Corriendo Ridge LOAO-CV...')

# Para pipelines con sample_weight, necesitamos pasar el argumento correcto
# sklearn Pipeline soporta: fit(X, y, model__sample_weight=w)
athletes = df_ml['cedula'].unique()
ridge_fold_maes = []
ridge_fold_info = []

for k, ath_test in enumerate(athletes):
    mask_train = df_ml['cedula'] != ath_test
    mask_test  = df_ml['cedula'] == ath_test

    X_tr = df_ml.loc[mask_train, FEATURES].values
    y_tr = df_ml.loc[mask_train, TARGET].values
    w_tr = df_ml.loc[mask_train, 'sample_weight'].values
    X_te = df_ml.loc[mask_test,  FEATURES].values
    y_te = df_ml.loc[mask_test,  TARGET].values

    ridge_pipe.fit(X_tr, y_tr, model__sample_weight=w_tr)
    y_pred = ridge_pipe.predict(X_te)
    mae    = mean_absolute_error(y_te, y_pred)
    ridge_fold_maes.append(mae)
    ridge_fold_info.append({'cedula': ath_test, 'mae': mae, 'n_test': mask_test.sum()})

    if (k + 1) % 10 == 0 or k == 0:
        print(f'  Fold {k+1}/{n_athletes} — MAE fold: {mae:.1f} sec/km')

ridge_mae_arr    = np.array(ridge_fold_maes)
ridge_mae_global = ridge_mae_arr.mean()
ridge_fold_df    = pd.DataFrame(ridge_fold_info)

print(f'\n=== Ridge LOAO-CV ===')
print(f'MAE global: {ridge_mae_global:.1f} ± {ridge_mae_arr.std():.1f} sec/km')
print(f'            ({ridge_mae_global/60:.2f} ± {ridge_mae_arr.std()/60:.2f} min/km)')
print(f'Mejora vs N1 (prior): {mae_n1_on_n2_data - ridge_mae_global:+.1f} sec/km',
      f'({(mae_n1_on_n2_data - ridge_mae_global)/mae_n1_on_n2_data*100:+.1f}%)')

In [ ]:
# ── 4.3 Visualización Ridge LOAO-CV ─────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# MAE por atleta (fold)
ax = axes[0]
sorted_fold = ridge_fold_df.sort_values('mae', ascending=False).reset_index(drop=True)
colors_bar = ['#d62728' if m > mae_n1_on_n2_data else 'steelblue' for m in sorted_fold['mae']]
ax.bar(range(len(sorted_fold)), sorted_fold['mae'], color=colors_bar, edgecolor='white')
ax.axhline(ridge_mae_global, color='navy', lw=2, linestyle='-', label=f'Ridge media = {ridge_mae_global:.1f}')
ax.axhline(mae_n1_on_n2_data, color='red', lw=2, linestyle='--', label=f'N1 prior = {mae_n1_on_n2_data:.1f}')
ax.set_xlabel('Atleta (fold) ordenado por MAE')
ax.set_ylabel('MAE (sec/km)')
ax.set_title('Ridge LOAO-CV — MAE por atleta\n(rojo = fold peor que N1 prior)')
ax.legend(fontsize=9)

# Distribución de MAE por fold
ax = axes[1]
ax.hist(ridge_mae_arr, bins=15, color='steelblue', edgecolor='white', alpha=0.8)
ax.axvline(ridge_mae_global, color='navy', lw=2, label=f'Media = {ridge_mae_global:.1f}')
ax.axvline(np.median(ridge_mae_arr), color='orange', lw=2, linestyle='--',
           label=f'Mediana = {np.median(ridge_mae_arr):.1f}')
ax.axvline(mae_n1_on_n2_data, color='red', lw=2, linestyle=':', label=f'N1 prior = {mae_n1_on_n2_data:.1f}')
ax.set_xlabel('MAE por fold (sec/km)')
ax.set_ylabel('# Atletas')
ax.set_title('Distribución de MAE por fold\n(Ridge LOAO-CV con sample weighting)')
ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig(FIGURES_DIR / '13_ridge_loao_cv.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 5. AutoML Manual — 7 Familias LOAO-CV

Protocolo equivalente al usado en N1 (NB11):
- **7 familias sklearn:** Ridge, Lasso, ElasticNet, GradientBoosting, RandomForest, LinearSVR, KNeighbors
- **Mismo LOAO-CV** (N folds, una por atleta elegible)
- **Mismos sample weights** en training
- **Métrica:** MAE sec/km por fold → vector de N valores para Friedman-Nemenyi

Hipótesis nula Friedman: todos los modelos tienen el mismo rango MAE (no hay diferencias significativas).

In [ ]:
# ── 5.1 Definición del zoo de modelos ────────────────────────────────────────
# Todos wrapped en Pipeline(scaler + model) para evitar data leakage
# Hyperparámetros conservadores para LOAO-CV rápida (N*7 fits)

MODEL_ZOO = {
    'Ridge':         Pipeline([('sc', StandardScaler()), ('model', Ridge(alpha=1.0))]),
    'Lasso':         Pipeline([('sc', StandardScaler()), ('model', Lasso(alpha=0.5, max_iter=2000))]),
    'ElasticNet':    Pipeline([('sc', StandardScaler()), ('model', ElasticNet(alpha=0.5, l1_ratio=0.5, max_iter=2000))]),
    'GradBoost':     GradientBoostingRegressor(n_estimators=150, max_depth=4, learning_rate=0.05,
                                               subsample=0.8, random_state=RANDOM_STATE),
    'RandomForest':  RandomForestRegressor(n_estimators=150, max_depth=10, min_samples_leaf=5,
                                           n_jobs=-1, random_state=RANDOM_STATE),
    'LinearSVR':     Pipeline([('sc', StandardScaler()), ('model', LinearSVR(C=10.0, max_iter=5000))]),
    'KNeighbors':    Pipeline([('sc', StandardScaler()), ('model', KNeighborsRegressor(n_neighbors=7))]),
}

print(f'Modelos en zoo: {list(MODEL_ZOO.keys())}')
print(f'Folds por modelo: {n_athletes}')
print(f'Total fits: {len(MODEL_ZOO) * n_athletes}')
print()
print('Nota: GradBoost y RF no soportan sample_weight via Pipeline — se pasan directamente.')
print('      Para los pipelines sklearn: fit(X, y, model__sample_weight=w)')

In [ ]:
# ── 5.2 LOAO-CV para todos los modelos ───────────────────────────────────────
import time

PIPELINE_MODELS = {'Ridge', 'Lasso', 'ElasticNet', 'LinearSVR'}  # KNN no soporta sample_weight
NATIVE_SW_MODELS = {'GradBoost', 'RandomForest'}   # soportan sample_weight directamente

all_results = {}     # model_name → dict con mae_per_fold, mae_global, etc.
X_all = df_ml[FEATURES].values
y_all = df_ml[TARGET].values
w_all = df_ml['sample_weight'].values
ced_all = df_ml['cedula'].values

for model_name, model in MODEL_ZOO.items():
    t0 = time.time()
    print(f'\n--- {model_name} ---')

    fold_maes = []
    for ath_test in athletes:
        train_mask = ced_all != ath_test
        test_mask  = ced_all == ath_test

        X_tr, y_tr, w_tr = X_all[train_mask], y_all[train_mask], w_all[train_mask]
        X_te, y_te        = X_all[test_mask],  y_all[test_mask]

        if model_name in PIPELINE_MODELS:
            model.fit(X_tr, y_tr, model__sample_weight=w_tr)
        elif model_name in NATIVE_SW_MODELS:
            model.fit(X_tr, y_tr, sample_weight=w_tr)
        else:
            model.fit(X_tr, y_tr)

        y_pred = model.predict(X_te)
        fold_maes.append(mean_absolute_error(y_te, y_pred))

    mae_arr = np.array(fold_maes)
    elapsed = time.time() - t0

    all_results[model_name] = {
        'mae_per_fold': mae_arr,
        'mae_global':   mae_arr.mean(),
        'mae_std':      mae_arr.std(),
        'mae_median':   np.median(mae_arr),
    }

    print(f'  MAE: {mae_arr.mean():.1f} ± {mae_arr.std():.1f} sec/km',
          f'({mae_arr.mean()/60:.2f} min/km)  [{elapsed:.0f}s]')

print('\n=== LOAO-CV completado para todos los modelos ===')

In [ ]:
# ── 5.3 Tabla comparativa de resultados ──────────────────────────────────────
rows = []
for name, res in all_results.items():
    rows.append({
        'Modelo':           name,
        'MAE (sec/km)':     round(res['mae_global'], 1),
        'Std (sec/km)':     round(res['mae_std'], 1),
        'MAE (min/km)':     round(res['mae_global'] / 60, 3),
        'Mediana (sec/km)': round(res['mae_median'], 1),
    })

# Agregar N1 prior como referencia
rows.append({
    'Modelo':           '— N1 Prior (referencia) —',
    'MAE (sec/km)':     round(mae_n1_on_n2_data, 1),
    'Std (sec/km)':     None,
    'MAE (min/km)':     round(mae_n1_on_n2_data / 60, 3),
    'Mediana (sec/km)': None,
})

results_df = pd.DataFrame(rows).sort_values('MAE (sec/km)').reset_index(drop=True)
print('=== COMPARATIVA LOAO-CV — 7 modelos + N1 prior ===')
print(results_df.to_string(index=False))

best_model_name = results_df.iloc[0]['Modelo']
best_mae        = results_df.iloc[0]['MAE (sec/km)']
mejora_vs_n1    = (mae_n1_on_n2_data - best_mae) / mae_n1_on_n2_data * 100

print(f'\n=== MODELO GANADOR: {best_model_name} ===')
print(f'MAE: {best_mae:.1f} sec/km ({best_mae/60:.3f} min/km)')
print(f'Mejora vs N1 prior: {mejora_vs_n1:+.1f}%')

In [ ]:
# ── 5.4 Visualización comparativa ────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

model_names  = list(all_results.keys())
mae_means    = [all_results[m]['mae_global'] for m in model_names]
mae_stds     = [all_results[m]['mae_std']    for m in model_names]
sorted_idx   = np.argsort(mae_means)
sorted_names = [model_names[i] for i in sorted_idx]
sorted_means = [mae_means[i] for i in sorted_idx]
sorted_stds  = [mae_stds[i]  for i in sorted_idx]

# Bar chart MAE
ax = axes[0]
colors_m = ['#2ecc71' if i == 0 else 'steelblue' for i in range(len(sorted_names))]
bars = ax.barh(sorted_names, sorted_means, xerr=sorted_stds,
               color=colors_m, edgecolor='white', capsize=4)
ax.axvline(mae_n1_on_n2_data, color='red', linestyle='--', lw=2,
           label=f'N1 prior = {mae_n1_on_n2_data:.1f}')
for bar, val, std in zip(bars, sorted_means, sorted_stds):
    ax.text(val + std + 0.5, bar.get_y() + bar.get_height()/2,
            f'{val:.1f}', va='center', fontsize=9)
ax.set_xlabel('MAE (sec/km) — menor es mejor')
ax.set_title('Comparativa LOAO-CV\n7 familias sklearn (con sample weighting)')
ax.legend(fontsize=9)

# Boxplot MAE por fold
ax = axes[1]
fold_data  = [all_results[m]['mae_per_fold'] for m in sorted_names]
bp = ax.boxplot(fold_data, labels=sorted_names, patch_artist=True, showfliers=True,
                vert=False)
for i, (patch, name) in enumerate(zip(bp['boxes'], sorted_names)):
    patch.set_facecolor('#2ecc71' if i == 0 else 'steelblue')
    patch.set_alpha(0.7)
ax.axvline(mae_n1_on_n2_data, color='red', linestyle='--', lw=2, label='N1 prior')
ax.set_xlabel('MAE por fold (sec/km)')
ax.set_title('Distribución de MAE por fold\n(cada punto = 1 atleta)')
ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig(FIGURES_DIR / '13_automl_comparativa.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 6. Prueba de Friedman-Nemenyi

La prueba de Friedman (Friedman 1940) es el equivalente no-paramétrico del ANOVA para diseños en bloque.  
Cada *bloque* es un atleta (fold); cada *tratamiento* es un modelo.  
**H0:** Todos los modelos tienen el mismo rango de MAE.  

Si H0 se rechaza (p < 0.05), se aplica la prueba post-hoc de Nemenyi para identificar qué pares de modelos son significativamente diferentes.

**Referencia:** Demšar J. (2006). Statistical comparisons of classifiers over multiple data sets. *JMLR*, 7, 1-30.

In [ ]:
# ── 6.1 Prueba de Friedman ────────────────────────────────────────────────────
model_names_ordered = list(all_results.keys())
fold_mae_matrix = np.column_stack([all_results[m]['mae_per_fold'] for m in model_names_ordered])
# Dimensiones: (n_athletes, n_models)

print(f'Matriz MAE: {fold_mae_matrix.shape} (folds x modelos)')
print(f'Filas = atletas, columnas = modelos')
print()

# Friedman test
stat, p_val = friedmanchisquare(*[fold_mae_matrix[:, j] for j in range(fold_mae_matrix.shape[1])])
print(f'=== Prueba de Friedman ===')
print(f'Estadístico chi²: {stat:.4f}')
print(f'p-value:          {p_val:.6f}')
print()

if p_val < 0.05:
    print('✅ H0 rechazada (p < 0.05): existen diferencias significativas entre modelos.')
    print('   Procedemos con post-hoc Nemenyi para identificar pares significativos.')
else:
    print('⚠️  H0 NO rechazada (p >= 0.05): no se detectan diferencias significativas.')
    print('   Los modelos tienen desempeño estadísticamente equivalente.')
print()

# Rangos medios (Friedman ranks) — menor rango = mejor modelo
from scipy.stats import rankdata
ranks_per_fold = np.apply_along_axis(rankdata, axis=1, arr=fold_mae_matrix)
mean_ranks = ranks_per_fold.mean(axis=0)
rank_df = pd.DataFrame({'Modelo': model_names_ordered, 'Rango medio': mean_ranks.round(3)})
rank_df = rank_df.sort_values('Rango medio').reset_index(drop=True)
print('Rangos medios de Friedman (menor = mejor):')
print(rank_df.to_string(index=False))

In [ ]:
# ── 6.2 Post-hoc Nemenyi ─────────────────────────────────────────────────────
if HAS_POSTHOCS and p_val < 0.05:
    # Nemenyi via scikit-posthocs
    nemenyi_mat = sp.posthoc_nemenyi_friedman(fold_mae_matrix)
    nemenyi_mat.columns = model_names_ordered
    nemenyi_mat.index   = model_names_ordered
    print('=== Matriz p-values Nemenyi ===')
    print(nemenyi_mat.round(4).to_string())
else:
    # Implementación manual del CD de Nemenyi (Demsar 2006, Eq 5)
    k = len(model_names_ordered)
    n = n_athletes
    # Valor critico qa para alpha=0.05: Tabla de Demsar (q_0.05 para k modelos)
    # qa_table[k-2] para k modelos:
    qa_table = [1.960, 2.344, 2.569, 2.728, 2.850, 2.949, 3.031, 3.102, 3.164]
    qa = qa_table[min(k - 2, len(qa_table) - 1)]
    cd = qa * np.sqrt(k * (k + 1) / (6.0 * n))
    print(f'=== Diferencia Critica de Nemenyi (CD) — Demsar 2006 ===')
    print(f'k = {k} modelos, N = {n} folds, alpha = 0.05')
    print(f'qa = {qa:.3f}')
    print(f'CD = {cd:.4f}')
    print()
    print('Diferencias de rango entre pares (significativo si |diff| > CD):')
    for i, m1 in enumerate(rank_df['Modelo']):
        r1 = rank_df.loc[rank_df['Modelo'] == m1, 'Rango medio'].values[0]
        for m2 in rank_df['Modelo'][i+1:]:
            r2 = rank_df.loc[rank_df['Modelo'] == m2, 'Rango medio'].values[0]
            diff = abs(r1 - r2)
            sig  = '(**)' if diff > cd else ''
            print(f'  {m1:15s} vs {m2:15s}: |diff| = {diff:.3f} {sig}')
    nemenyi_mat = None

In [ ]:
# ── 6.3 Critical Difference diagram (Demsar plot) ────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Rangos medios — bar chart
ax = axes[0]
bar_colors = ['#2ecc71'] + ['steelblue'] * (len(rank_df) - 1)
bars = ax.barh(rank_df['Modelo'], rank_df['Rango medio'],
               color=bar_colors, edgecolor='white')
for bar, val in zip(bars, rank_df['Rango medio']):
    ax.text(bar.get_width() + 0.02, bar.get_y() + bar.get_height()/2,
            f'{val:.2f}', va='center', fontsize=9)
ax.set_xlabel('Rango medio Friedman (menor = mejor)')
ax.set_title(f'Prueba de Friedman\nchi² = {stat:.2f}, p = {p_val:.4f}')

# Nemenyi heatmap (si disponible) o distribución de MAE por fold
ax = axes[1]
if nemenyi_mat is not None:
    # Heatmap de significancia
    sig_mat = (nemenyi_mat < 0.05).astype(float)
    sns.heatmap(sig_mat.loc[rank_df['Modelo'], rank_df['Modelo']],
                annot=nemenyi_mat.loc[rank_df['Modelo'], rank_df['Modelo']].round(3),
                fmt='.3f', cmap='RdYlGn_r', ax=ax, vmin=0, vmax=0.2,
                annot_kws={'size': 8})
    ax.set_title('Post-hoc Nemenyi p-values\n(verde = no significativo, rojo = sig. p<0.05)')
else:
    # Violinplot de MAE por modelo
    fold_data_ranked = [all_results[m]['mae_per_fold'] for m in rank_df['Modelo']]
    vp = ax.violinplot(fold_data_ranked, vert=False, showmedians=True)
    ax.set_yticks(range(1, len(rank_df) + 1))
    ax.set_yticklabels(rank_df['Modelo'])
    ax.axvline(mae_n1_on_n2_data, color='red', lw=2, linestyle='--', label='N1 prior')
    ax.set_xlabel('MAE por fold (sec/km)')
    ax.set_title('Distribución de MAE por fold\n(violinplot — 1 punto = 1 atleta)')
    ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig(FIGURES_DIR / '13_friedman_nemenyi.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 7. Modelo Ganador — Análisis y Serialización

In [ ]:
# ── 7.1 Resumen ejecutivo de resultados ──────────────────────────────────────
winner_name = rank_df.iloc[0]['Modelo']
winner_mae  = all_results[winner_name]['mae_global']
winner_std  = all_results[winner_name]['mae_std']

print('=' * 60)
print('RESUMEN EJECUTIVO — Nivel 2 LOAO-CV')
print('=' * 60)
print(f'Cohorte RUNA: {n_athletes} atletas, {n_sessions:,} sesiones')
print(f'Período cubierto: 3 años (historial Strava completo)')
print()
print(f'Baseline N1 (prior poblacional):  {mae_n1_on_n2_data:.1f} sec/km ({mae_n1_on_n2_data/60:.3f} min/km)')
print(f'Ridge N2 LOAO-CV:                 {ridge_mae_global:.1f} sec/km ({ridge_mae_global/60:.3f} min/km)')
print(f'Modelo ganador ({winner_name}):   {winner_mae:.1f} ± {winner_std:.1f} sec/km ({winner_mae/60:.3f} min/km)')
print()
mejora_ridge = (mae_n1_on_n2_data - ridge_mae_global) / mae_n1_on_n2_data * 100
mejora_best  = (mae_n1_on_n2_data - winner_mae)        / mae_n1_on_n2_data * 100
print(f'Mejora Ridge N2 vs N1:    {mejora_ridge:+.1f}%')
print(f'Mejora {winner_name} N2 vs N1: {mejora_best:+.1f}%')
print()
print(f'Prueba de Friedman: chi² = {stat:.2f}, p = {p_val:.4f}')
if p_val < 0.05:
    print('  → Diferencias significativas entre modelos (alpha=0.05)')
else:
    print('  → Modelos estadisticamente equivalentes (alpha=0.05)')
    print('  → Recomendacion: usar Ridge por parsimonia e interpretabilidad')
print('=' * 60)

In [ ]:
# ── 7.2 Feature importance / coeficientes del modelo ganador ─────────────────
# Reentrenar ganador con TODO el dataset (todos los atletas) para serializar
winner_model = MODEL_ZOO[winner_name]
w_final = df_ml['sample_weight'].values

if winner_name in PIPELINE_MODELS:
    winner_model.fit(X_all, y_all, model__sample_weight=w_final)
elif winner_name in NATIVE_SW_MODELS:
    winner_model.fit(X_all, y_all, sample_weight=w_final)
else:
    winner_model.fit(X_all, y_all)

print(f'Modelo ganador reentrenado con {n_sessions:,} sesiones ({n_athletes} atletas).')

# Extraer importancias / coeficientes
fig, ax = plt.subplots(figsize=(9, 6))

if hasattr(winner_model, 'feature_importances_'):
    fi = pd.Series(winner_model.feature_importances_, index=FEATURES).sort_values(ascending=True)
    fi.plot(kind='barh', ax=ax, color='steelblue', edgecolor='white')
    ax.set_title(f'{winner_name} — Feature Importance')
    ax.set_xlabel('Importancia relativa')
elif hasattr(winner_model, 'named_steps'):
    m = winner_model.named_steps['model']
    if hasattr(m, 'coef_'):
        coef = pd.Series(np.abs(m.coef_), index=FEATURES).sort_values(ascending=True)
        coef.plot(kind='barh', ax=ax, color='steelblue', edgecolor='white')
        ax.set_title(f'{winner_name} — |Coeficientes| (features estandarizadas)')
        ax.set_xlabel('|Coeficiente|')
        # Dirección del efecto
        coef_dir = pd.Series(m.coef_, index=FEATURES).sort_values()
        print('\nDirección de coeficientes (top 5 positivos = aumentan el pace):')
        print(coef_dir.tail(5).round(2))
        print('Top 5 negativos (reducen el pace = más rápido):')
        print(coef_dir.head(5).round(2))
else:
    ax.text(0.5, 0.5, 'Feature importance no disponible para este modelo',
            ha='center', va='center', transform=ax.transAxes)

plt.tight_layout()
plt.savefig(FIGURES_DIR / '13_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── 7.3 Calibración del ganador — predicted vs actual ────────────────────────
# Usar los MAE por fold del ganador (out-of-fold predictions)
oof_preds = np.full(len(df_ml), np.nan)

for ath_test in athletes:
    train_mask = ced_all != ath_test
    test_mask  = ced_all == ath_test

    X_tr, y_tr, w_tr = X_all[train_mask], y_all[train_mask], w_all[train_mask]
    X_te = X_all[test_mask]

    if winner_name in PIPELINE_MODELS:
        winner_model.fit(X_tr, y_tr, model__sample_weight=w_tr)
    elif winner_name in NATIVE_SW_MODELS:
        winner_model.fit(X_tr, y_tr, sample_weight=w_tr)
    else:
        winner_model.fit(X_tr, y_tr)

    oof_preds[test_mask] = winner_model.predict(X_te)

# Reentrenar final
if winner_name in PIPELINE_MODELS:
    winner_model.fit(X_all, y_all, model__sample_weight=w_final)
elif winner_name in NATIVE_SW_MODELS:
    winner_model.fit(X_all, y_all, sample_weight=w_final)
else:
    winner_model.fit(X_all, y_all)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

y_true_min = y_all / 60
y_pred_min = oof_preds / 60
y_n1_min   = df_ml['pred_nivel1'].values / 60

# Predicted vs actual — modelo ganador
ax = axes[0]
ax.scatter(y_true_min, y_pred_min, alpha=0.15, s=6, color='steelblue')
lims = [y_true_min.min()-0.3, y_true_min.max()+0.3]
ax.plot(lims, lims, 'k--', lw=1.5, label='Perfecta')
ax.set_xlabel('Pace real (min/km)')
ax.set_ylabel('Pace predicho (min/km)')
ax.set_title(f'{winner_name} N2 — OOF\nMAE = {winner_mae:.1f} sec/km')
ax.legend(fontsize=9)

# Predicted vs actual — N1 prior
ax = axes[1]
ax.scatter(y_true_min, y_n1_min, alpha=0.15, s=6, color='tomato')
ax.plot(lims, lims, 'k--', lw=1.5)
ax.set_xlabel('Pace real (min/km)')
ax.set_ylabel('N1 predicho (min/km)')
ax.set_title(f'N1 Prior (referencia)\nMAE = {mae_n1_on_n2_data:.1f} sec/km')

# Residuos — ganador vs N1
ax = axes[2]
resid_winner = (oof_preds - y_all) / 60
resid_n1     = (df_ml['pred_nivel1'].values - y_all) / 60
ax.hist(resid_winner, bins=60, alpha=0.7, color='steelblue', label=winner_name, density=True)
ax.hist(resid_n1, bins=60, alpha=0.5, color='tomato', label='N1 prior', density=True)
ax.axvline(0, color='black', lw=1.5, linestyle='--')
ax.set_xlabel('Residuo (min/km) — 0 = perfecto')
ax.set_ylabel('Densidad')
ax.set_title('Distribución de residuos\n(ganador vs N1)')
ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig(FIGURES_DIR / '13_calibracion_ganador.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── 7.4 Serializar modelo N2 ─────────────────────────────────────────────────
import pickle

MODEL_SAVE_DIR = Path('../../api/models')
MODEL_SAVE_DIR.mkdir(exist_ok=True)

n2_payload = {
    'model':           winner_model,
    'model_name':      winner_name,
    'features':        FEATURES,
    'target':          TARGET,
    'n_athletes':      int(n_athletes),
    'n_sessions':      int(n_sessions),
    'mae_loao_cv':     float(winner_mae),
    'mae_std':         float(winner_std),
    'mae_n1_baseline': float(mae_n1_on_n2_data),
    'mejora_vs_n1_pct': float(mejora_best),
    'friedman_chi2':   float(stat),
    'friedman_pval':   float(p_val),
    'vdot_median_imputed': float(vdot_median),
    'n2_thresholds':   {'min_hr_runs': N2_MIN_RUNS, 'min_weeks': N2_MIN_WEEKS},
    'trained_at':      datetime.now().isoformat(),
    'notes': (
        'N2 — Prior de cohorte RUNA. Entrenado con LOAO-CV. '
        'Sample weight = 1/n_sesiones_atleta. '
        'Usa pred_nivel1 como stacking feature (Wolpert 1992). '
        'CTL/ATL excluidos (→ N3). PRs solo via VDOT (no como feature directa).'
    ),
}

save_path = MODEL_SAVE_DIR / 'nivel2_v1.pkl'
with open(save_path, 'wb') as f:
    pickle.dump(n2_payload, f)

print(f'Modelo N2 serializado: {save_path}')
print(f'  Modelo: {winner_name}')
print(f'  Features: {len(FEATURES)}')
print(f'  MAE LOAO-CV: {winner_mae:.1f} sec/km')
print(f'  Mejora vs N1: {mejora_best:+.1f}%')

In [ ]:
# ── 7.5 Guardar resultados LOAO-CV en JSON (para tesis) ──────────────────────
results_json = {
    'nb': 'NB13',
    'title': 'Nivel 2 — Prior de Cohorte RUNA (LOAO-CV)',
    'date': datetime.now().isoformat()[:10],
    'cohorte': {'n_athletes': int(n_athletes), 'n_sessions': int(n_sessions)},
    'eligibility': {'min_hr_runs': N2_MIN_RUNS, 'min_weeks': N2_MIN_WEEKS},
    'n1_baseline_mae_sec_km': round(mae_n1_on_n2_data, 2),
    'ridge_n2_mae_sec_km':   round(ridge_mae_global, 2),
    'models': {
        name: {
            'mae_global':  round(r['mae_global'], 2),
            'mae_std':     round(r['mae_std'], 2),
            'mae_min':     round(r['mae_per_fold'].min(), 2),
            'mae_max':     round(r['mae_per_fold'].max(), 2),
            'mae_median':  round(r['mae_median'], 2),
            'friedman_rank': round(mean_ranks[model_names_ordered.index(name)], 3),
        }
        for name, r in all_results.items()
    },
    'friedman': {'chi2': round(float(stat), 4), 'p_value': round(float(p_val), 6)},
    'winner': {
        'model_name':      winner_name,
        'mae_sec_km':      round(winner_mae, 2),
        'mae_std_sec_km':  round(winner_std, 2),
        'mejora_vs_n1_pct': round(mejora_best, 2),
    },
    'features': FEATURES,
}

json_path = Path('../../api/models/nivel2_results_nb13.json')
with open(json_path, 'w', encoding='utf-8') as f:
    json.dump(results_json, f, indent=2, ensure_ascii=False)

print(f'Resultados guardados en: {json_path}')
print(json.dumps(results_json, indent=2, ensure_ascii=False))

---
## 8. Conclusiones

### Hallazgos principales

| Dimensión | Resultado |
|---|---|
| **Cohorte** | N atletas N2 elegibles, N sesiones con HR |
| **Baseline N1** | `mae_n1_on_n2_data` sec/km (prior poblacional Endomondo) |
| **Ridge N2 LOAO-CV** | `ridge_mae_global` ± `ridge_mae_global.std` sec/km |
| **Ganador LOAO-CV** | Ver celda 7.5 |
| **Prueba Friedman** | Chi²=X, p=Y (ver celda 6.1) |
| **Validación** | LOAO-CV — leave-one-athlete-out |

### Interpretación

1. **El prior N1 como stacking feature es útil:** `pred_nivel1` captura la relación FC↔ritmo poblacional y es un feature fuerte para N2, permitiendo aprovechar 20,710 sesiones de Endomondo en el modelo de cohorte.

2. **LOAO-CV simula fielmente el caso de uso real:** Un atleta nuevo (fold de test) nunca fue visto en entrenamiento, replicando el escenario de onboarding donde no hay historial propio disponible todavía.

3. **Sample weighting es esencial:** Sin w=1/n_sesiones, atletas con >200 sesiones dominarían el loss. El weighting garantiza que cada atleta contribuya equitativamente al aprendizaje.

4. **Conexión con N3:** El Nivel 3 extenderá N2 agregando CTL/ATL/TSB como features (datos que solo están disponibles después de semanas de historial), y carreras reales como ground-truth adicional.

### Aporte a la tesis

> *"Un modelo de la cohorte RUNA (N atletas, N sesiones) entrenado con LOAO-CV logra MAE = X sec/km, representando una mejora de Y% respecto al prior poblacional N1 (Z sec/km). La prueba de Friedman (chi²=A, p=B) confirma/no confirma diferencias estadísticas entre las 7 familias evaluadas."*

### Próximos pasos
- **NB14:** Nivel 3 longitudinal — agregar CTL/ATL/TSB como features adicionales
- **NB15:** Integración del sistema jerárquico completo (N1 → N2 → N3)
- **Producción:** Integrar `nivel2_v1.pkl` en `api/routers/athletes.py` → endpoint `/prediction`